 <center>

# Compare CNN Models on CIFAR-10 using Pytorch

**Name:** Marrion Kiprop Cherop  
**Registration Number:** ST62/80971/2024  
**Programme:** Master of Science in Artificial Intelligence  
**Course:** CSA 809 — Deep Learning
**Module:** Module 6 Normalization in CNN  

 <center>


## Introduction

This notebook builds two CNNs on CIFAR-10 to isolate the effect of BatchNormalization and Dropout under a short, realistic training budget of upto 2 epochs. Model A is a plain two-layer convolutional network with no regularization. Model B adds BatchNorm after each convolution and Dropout(0.5) before the output layer, everything else held identical, so any difference in behaviour traces back to those two additions and not to some other architectural change. The comparison tracked is training accuracy against validation accuracy, since the gap between the two, not either number alone, is what signals overfitting, alongside training time per epoch, since BatchNorm and Dropout both add computation the plain model doesn't pay for.


## 1. Imports

Standard PyTorch and torchvision imports for building, training, and loading CIFAR-10.


In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cpu


## 2. Load CIFAR-10

Images are converted to tensors and normalized per channel. The mean and std values below are the commonly used CIFAR-10 per-channel statistics; normalizing against the dataset's own actual distribution (rather than a generic 0.5/0.5) keeps the input scale closer to what the network expects, and matters more here than usual because Model B's BatchNorm layers are themselves computing per-batch statistics on top of this input, so starting from a well-scaled input reduces the number of things BatchNorm has to correct for in the first pass.


In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

train_set = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

print(f"Train samples: {len(train_set)}, Test samples: {len(test_set)}")


/home/langstkip/workspace/langstEnv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train samples: 50000, Test samples: 10000


## 3. Model A — Basic CNN (no BatchNorm, no Dropout)

Two convolutional layers, each followed by ReLU and max pooling, feeding into two fully connected layers. `padding=1` on both convs keeps spatial size unchanged through convolution, so the two `MaxPool2d(2,2)` calls are what take the input from 32x32 down to 8x8, which is what makes the given `Linear(64*8*8, 128)` shape correct. This model has no mechanism to resist overfitting beyond whatever the optimizer and short epoch count naturally limit.


In [3]:
class CNN_A(nn.Module):
    def __init__(self):
        super(CNN_A, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))   # 16x16 -> 8x8
        x = x.view(x.size(0), -1)              # flatten to 64*8*8
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


## 4. Model B — CNN with BatchNorm + Dropout

Same two convolutional layers as Model A, but each is followed by `BatchNorm2d`, which normalizes activations per channel using batch statistics before the ReLU nonlinearity. A `Dropout(0.5)` layer sits before the final output layer, randomly zeroing half the incoming units during training. Everything else, kernel sizes, channel counts, pooling, fully connected layer sizes, is identical to Model A, so this model isolates what BatchNorm and Dropout contribute rather than mixing in unrelated architectural changes.


In [4]:
class CNN_B(nn.Module):
    def __init__(self):
        super(CNN_B, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 32x32 -> 16x16
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 16x16 -> 8x8
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


## 5. Training and evaluation routines

`train_one_epoch` runs the model in training mode, meaning Dropout is active and BatchNorm uses current-batch statistics. `evaluate` switches to `model.eval()`, so Dropout is disabled and BatchNorm switches to its running mean/variance estimated across training, which is the correct behaviour for measuring genuine validation performance rather than training-mode behaviour on the test set.


In [5]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


## 6. Running both models

Both models train for the same number of epochs, same optimizer (Adam), same learning rate, and same batch size, so any difference in the results traces back to BatchNorm and Dropout rather than an uncontrolled variable.


In [6]:
def run_experiment(model_class, epochs=2, lr=1e-3):
    model = model_class().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []
    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, test_loader, criterion)
        elapsed = time.time() - start

        history.append({
            'epoch': epoch,
            'train_loss': train_loss, 'train_acc': train_acc,
            'val_loss': val_loss, 'val_acc': val_acc,
            'time_sec': elapsed
        })
        print(f"Epoch {epoch}/{epochs} | "
              f"train_acc: {train_acc:.4f} val_acc: {val_acc:.4f} | "
              f"train_loss: {train_loss:.4f} val_loss: {val_loss:.4f} | "
              f"time: {elapsed:.1f}s")
    return model, history


print("=== Model A: Basic CNN (no BN, no Dropout) ===")
model_a, history_a = run_experiment(CNN_A, epochs=2)

print()
print("=== Model B: CNN with BatchNorm + Dropout ===")
model_b, history_b = run_experiment(CNN_B, epochs=2)


=== Model A: Basic CNN (no BN, no Dropout) ===
Epoch 1/2 | train_acc: 0.5391 val_acc: 0.6452 | train_loss: 1.2908 val_loss: 1.0005 | time: 41.8s
Epoch 2/2 | train_acc: 0.6838 val_acc: 0.6971 | train_loss: 0.9032 val_loss: 0.8713 | time: 24.8s

=== Model B: CNN with BatchNorm + Dropout ===
Epoch 1/2 | train_acc: 0.3913 val_acc: 0.5324 | train_loss: 1.6288 val_loss: 1.3032 | time: 23.8s
Epoch 2/2 | train_acc: 0.4834 val_acc: 0.6204 | train_loss: 1.3888 val_loss: 1.0939 | time: 27.7s


## 7. Comparison

For each model, the final epoch's train accuracy minus validation accuracy is the overfitting gap. A larger gap means the model is fitting the training set more than it is generalizing. Total training time reflects the extra computation BatchNorm's per-channel statistics and Dropout's masking add on top of the plain model.


In [7]:
print("=== Comparison Summary ===")
for label, history in [('Model A', history_a), ('Model B', history_b)]:
    final = history[-1]
    gap = final['train_acc'] - final['val_acc']
    total_time = sum(h['time_sec'] for h in history)
    print(f"{label}: final train_acc={final['train_acc']:.4f}, "
          f"val_acc={final['val_acc']:.4f}, "
          f"train/val gap={gap:.4f}, "
          f"total_time={total_time:.1f}s")


=== Comparison Summary ===
Model A: final train_acc=0.6838, val_acc=0.6971, train/val gap=-0.0133, total_time=66.6s
Model B: final train_acc=0.4834, val_acc=0.6204, train/val gap=-0.1370, total_time=51.5s


## Explanation

At only 2 epochs, we don't expect Model B to show a higher validation accuracy than Model A. BatchNorm and Dropout both slow down how quickly a network fits its training data by design, BatchNorm by normalizing activations against batch statistics that shift as weights update, Dropout(0.5) by discarding half the units feeding the output layer on every training step. A short training budget favours whichever model can fit fastest, which is usually the plain model, not the regularized one.

The number that actually matters here is the train and validation gap. At 2 epochs neither model has trained long enough to overfit meaningfully, we don't expect Model A's gap to look bigger yet either. This short run helps establish early trend, not determining which model generalizes better. The training accuracy for the basic model - Model A is at 68% with a validation accuracy if 69% while the regularized model B is at 48% training accuracy and validation accuracy of 62%. 
